# Actividad (Parte 5) Explicabilidad usando contraejemplos
## Materia: Inteligencia Artificial Explicable - MSc. Inteligencia Artificial
Author: Esteban García-Cuesta, Departamento de Inteligencia Artificial, UPM (License CC-BY-NC)

This code has been developed to be used exclusively for educational purposes.

## Introducción
La explicabilidad con contraejemplos permite conocer el funcionamiento del modelo modificando las entradas e identificar las opciones posibles para revertir una decisión no favorable. En esta actividad utilizaras la base de datos Heart Disease UCI Machine Learning Repository.

## Objetivos:
  - Aprender como obtener un contraejemplo dado un caso
  - Aprender a obtener un mejor contraejemplo usando el parámetro $\lambda$ de la fórmula de Watcher $\lambda (f_{\theta}(x')-y')+d(x^e-x')$
  - Aprender a aplicar en un caso de reversión de decisión las técnicas de contraejemplo e interpretar los resultados

## Para hacer
  - Realiza los cambios en el código necesarios de acuerdo a las instrucciones de la actividad y responde a las preguntas que se indican en dichas instrucciones.

In [2]:
# Instalación de la librerio ucimlrepo
!pip install ucimlrepo

In [3]:
#Lectura de los datos desde el repositorio UCI
from ucimlrepo import fetch_ucirepo
import pandas as pd

# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
dfX = heart_disease.data.features
dfY = heart_disease.data.targets
df = pd.concat([dfX, dfY], axis=1)
df = df.dropna()
dfX = df[['age','trestbps','chol','thalach','oldpeak','ca']]
dfY = df['num']
dfX.head()


,age,trestbps,chol,thalach,oldpeak,ca
0,63,145,233,150,2.3,0.0
1,67,160,286,108,1.5,3.0
2,67,120,229,129,2.6,2.0
3,37,130,250,187,3.5,0.0
4,41,130,204,172,1.4,0.0


In [4]:
# Aprendizaje del modelo random forest

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X = dfX.values
y = np.ravel(dfY.values)

clf = RandomForestClassifier(random_state=0)
clf.fit(X, y)

RandomForestClassifier(random_state=0)

In [6]:
# Predicción para el ejemplo 55
from mlxtend.evaluate import create_counterfactual

x_ref = X[55]

print('True label:', y[55])
print('Predicted label:', clf.predict(x_ref.reshape(1, -1))[0])
print('Predicted probas:', clf.predict_proba(x_ref.reshape(1, -1)))

True label: 1
Predicted label: 1
Predicted probas: [[0.02 0.72 0.07 0.19 0.  ]]


In [ ]:
# Búsqueda de un contraejemplo con un lammbda fijo

res = create_counterfactual(x_reference=x_ref,
                            y_desired=0,
                            model=clf,
                            X_dataset=X,
                            y_desired_proba=1,
                            lammbda=1, #  hyperparameter
                            random_seed=123)

print('Features of the 55th example:', x_ref)
print('Features of the countefactual:', res)

print('Predictions for counterfactual:\n')
print('Predicted label:', clf.predict(res.reshape(1, -1))[0])
print('Predicted probas:', clf.predict_proba(res.reshape(1, -1)))

Features of the 55th example: [ 54.  124.  266.  109.    2.2   1. ]
Features of the countefactual: [ 5.39803423e+01  1.23999991e+02  2.65999967e+02  1.09500107e+02
 -3.92200090e-04  9.64418746e-04]
Predictions for counterfactual:

Predicted label: 0
Predicted probas: [[0.57 0.25 0.05 0.13 0.  ]]


## Apartado a

In [ ]:
# Para elimianr la notacion cientifica de los array (e+0...)
np.set_printoptions(suppress=True)

lammbda_values = np.arange(1, 20+1)
class_study = 0
precision_for_class = 0.8

print('Features of the 55th example:', x_ref)
for lammbda in lammbda_values:
  print('\n\tLammbda value: ', lammbda)
  res = create_counterfactual(x_reference=x_ref, y_desired=class_study, model=clf, X_dataset=X, y_desired_proba=1,
                              lammbda=lammbda, random_seed=123)

  print('\tFeatures of the countefactual:', res)

  print('\tPredictions for counterfactual:\n')
  prediction = clf.predict(res.reshape(1, -1))[0]
  prediction_proba = clf.predict_proba(res.reshape(1, -1))
  print('\t\tPredicted label:', prediction)
  print('\t\tPredicted probas:', prediction_proba)

  if prediction_proba[0][class_study] > precision_for_class:
    print('\n\tFound counterfactual, that probability to be in class {0} greater than {1}'.format(class_study, precision_for_class))
    break

Features of the 55th example: [ 54.  124.  266.  109.    2.2   1. ]

	Lammbda value:  1
	Features of the countefactual: [ 53.98034227 123.99999127 265.9999669  109.50010668  -0.0003922
   0.00096442]
	Predictions for counterfactual:

		Predicted label: 0
		Predicted probas: [[0.57 0.25 0.05 0.13 0.  ]]

	Lammbda value:  2
	Features of the countefactual: [ 54.00000144 123.99999891 265.99996751 154.50001807  -0.00077225
   0.00034379]
	Predictions for counterfactual:

		Predicted label: 0
		Predicted probas: [[0.85 0.13 0.   0.02 0.  ]]

	Found counterfactual, that probability to be in class 0 greater than 0.8


## Apartado b


In [9]:
# Para elimianr la notacion cientifica de los array (e+0...)
np.set_printoptions(suppress=True)

x_ref_2 = X[2]
x_ref_55 = X[55]
x_ref_56 = X[56]

lammbda_values = np.arange(20, 20+1)
class_study = 0
precision_for_class = 0.5

print('\nFeatures of the 2th example:', x_ref_2)
print('True label:', y[2])
for lammbda in lammbda_values:
  print('\n\tLammbda value: ', lammbda)
  res_2 = create_counterfactual(x_reference=x_ref_2, y_desired=class_study, model=clf, X_dataset=X, y_desired_proba=precision_for_class,
                            lammbda=lammbda, random_seed=123)

  print('\tFeatures of the countefactual:', res_2)

  print('\tPredictions for counterfactual:\n')
  prediction = clf.predict(res_2.reshape(1, -1))[0]
  prediction_proba = clf.predict_proba(res_2.reshape(1, -1))
  print('\t\tPredicted label:', prediction)
  print('\t\tPredicted probas:', prediction_proba)

  if prediction_proba[0][class_study] > precision_for_class:
    print('\n\tFound counterfactual, that probability to be in class {0} greater than {1}'.format(class_study, precision_for_class))
    break


print('\nFeatures of the 55th example:', x_ref_55)
print('True label:', y[55])
for lammbda in lammbda_values:
  print('\n\tLammbda value: ', lammbda)
  res_55 = create_counterfactual(x_reference=x_ref_55, y_desired=class_study, model=clf, X_dataset=X, y_desired_proba=precision_for_class,
                            lammbda=lammbda, random_seed=123)

  print('\tFeatures of the countefactual:', res_55)

  print('\tPredictions for counterfactual:\n')
  prediction = clf.predict(res_55.reshape(1, -1))[0]
  prediction_proba = clf.predict_proba(res_55.reshape(1, -1))
  print('\t\tPredicted label:', prediction)
  print('\t\tPredicted probas:', prediction_proba)

  if prediction_proba[0][class_study] > precision_for_class:
    print('\n\tFound counterfactual, that probability to be in class {0} greater than {1}'.format(class_study, precision_for_class))
    break

print('\nFeatures of the 56th example:', x_ref_56)
print('True label:', y[56])
for lammbda in lammbda_values:
  print('\n\tLammbda value: ', lammbda)
  res_56 = create_counterfactual(x_reference=x_ref_56, y_desired=class_study, model=clf, X_dataset=X, y_desired_proba=precision_for_class,
                            lammbda=lammbda, random_seed=123)

  print('\tFeatures of the countefactual:', res_56)

  print('\tPredictions for counterfactual:\n')
  prediction = clf.predict(res_56.reshape(1, -1))[0]
  prediction_proba = clf.predict_proba(res_56.reshape(1, -1))
  print('\t\tPredicted label:', prediction)
  print('\t\tPredicted probas:', prediction_proba)

  if prediction_proba[0][class_study] > precision_for_class:
    print('\n\tFound counterfactual, that probability to be in class {0} greater than {1}'.format(class_study, precision_for_class))
    break


Features of the 2th example: [ 67.  120.  229.  129.    2.6   2. ]
True label: 1

	Lammbda value:  20
	Features of the countefactual: [ 55.49990451 119.9500287  229.00001047 128.99995713   0.00034654
   0.00021215]
	Predictions for counterfactual:

		Predicted label: 0
		Predicted probas: [[0.65 0.26 0.02 0.07 0.  ]]

	Found counterfactual, that probability to be in class 0 greater than 0.5

Features of the 55th example: [ 54.  124.  266.  109.    2.2   1. ]
True label: 1

	Lammbda value:  20
	Features of the countefactual: [ 54.0000036  123.99999837 230.99922765 194.53659005  -0.00070917
   0.00017306]
	Predictions for counterfactual:

		Predicted label: 0
		Predicted probas: [[0.9  0.06 0.   0.02 0.02]]

	Found counterfactual, that probability to be in class 0 greater than 0.5

Features of the 56th example: [ 50.  140.  233.  163.    0.6   1. ]
True label: 1

	Lammbda value:  20
	Features of the countefactual: [ 51.5000023  135.92354564 232.49815079 141.41810341   0.00045592
   0.00